# Kaggle Inference — Qwen2.5-1.5B-Instruct (Data Pipeline Incident Response)

Runs the schema-drift agent using **Qwen/Qwen2.5-1.5B-Instruct** (this is the HuggingFace repo name for the Qwen3-VL 4B model).

**VRAM budget on Kaggle T4 (15 GB):**
- Model weights in 4-bit NF4: ~2.5 GB
- Vision encoder (resident even for text tasks): ~0.8 GB
- KV cache + activations (max_new_tokens=512, history=16 turns): ~1.5 GB
- Framework overhead: ~0.4 GB
- **Total: ~5.2 GB** — leaves 9+ GB headroom, very safe.

---

## Kaggle Secrets required

Add these in **Settings → Secrets** before running:

| Secret name | What it is | Minimum permission |
|---|---|---|
| `GITHUB_TOKEN` | Classic Personal Access Token from github.com/settings/tokens | `repo` scope (needed to clone a private repo). If the repo is public, only `public_repo` is enough. |
| `HF_TOKEN` | HuggingFace token from huggingface.co/settings/tokens | **Read** access. If `Qwen/Qwen2.5-1.5B-Instruct` is gated, you also need to accept the model license on the HF model page once while logged in — the token itself stays read-only. |

> **HF_TOKEN write access is NOT needed for inference.** Only needed later if you push a fine-tuned model to the Hub (training notebook).


In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient

_s           = UserSecretsClient()
GITHUB_TOKEN = _s.get_secret('GITHUB_TOKEN')
HF_TOKEN     = _s.get_secret('HF_TOKEN')

os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/standing-on-giants/Meta_hackathon.git'
REPO_DIR = '/kaggle/working/Meta_hackathon'
BRANCH   = 'dev/pratham'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git fetch --all --quiet')

os.system(f'cd {REPO_DIR} && git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}')
os.system(f'cd {REPO_DIR} && git pull origin {BRANCH} --quiet')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
# Force-reload src modules so changes from git pull take effect
import importlib, sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]
print('Repo ready:', os.getcwd())
print('Current branch:', BRANCH)


In [10]:
# Install deps — qwen-vl-utils handles the Qwen2.5-VL image/video preprocessing pipeline
# We only do text inference here but the package is required by the model class
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'bitsandbytes',
    'qwen-vl-utils', 'pandas', 'numpy', 'openai', 'python-dotenv'], check=True)

import bitsandbytes as bnb, torch
print(f'bitsandbytes : {bnb.__version__}')
print(f'torch        : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


bitsandbytes : 0.49.2
torch        : 2.10.0+cu128
CUDA         : True
GPU          : Tesla T4
VRAM         : 15.6 GB


In [ ]:
import torch
# warnings and missing vision weights. The correct class ships in
# transformers>=4.49.0 — check the version printed above.
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
except ImportError:
    # transformers < 4.49: fall back to the older class
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.eval()

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded : {MODEL_ID}')
print(f'VRAM alloc   : {allocated:.2f} GB  reserved: {reserved:.2f} GB  free: {total-reserved:.2f} GB')


In [ ]:
import re, json, textwrap
from typing import Any, Dict, List, Optional
from src.models import PipelineAction, PipelineObservation

MODEL_NAME  = MODEL_ID
BENCHMARK   = 'data_pipeline_incident_response'

# ── These are the values the runner uses — change them here ────────
# MAX_TOKENS controls how long each model response can be.
# 512 was too small and caused JSON to be truncated mid-output.
# 1024 matches the Gemini script default and is safe on T4.
MAX_TOKENS  = int(os.getenv('MAX_TOKENS',   '1024'))
TEMPERATURE = float(os.getenv('TEMPERATURE', '0.1'))

# MAX_STEPS controls the episode length passed to run_episode().
# The env has its own obs.max_steps (hardcoded 20 inside DataPipelineEnv).
# We pass OUR max_steps to the loop AND surface it in the prompt as
# STEP {step}/{max_steps} — NOT obs.max_steps — so the model always
# sees the correct budget regardless of what the env reports internally.
MAX_STEPS   = int(os.getenv('MAX_STEPS', '100'))

SUCCESS_SCORE_THRESHOLD = 0.1

FALLBACK_ACTION = PipelineAction(action_type='compare_schema', params={'table': 'raw_ads_insights'})


# ── OpenEnv stdout logging — do not modify format ──────────────────
def log_start(task: str, env: str, model: str) -> None:
    print(f'[START] task={task} env={env} model={model}', flush=True)

def log_step(step: int, action: str, reward: float, done: bool, error: Optional[str]) -> None:
    error_val   = error if error else 'null'
    action_safe = action.replace('\n', ' ').replace('\r', '')[:200]
    print(
        f'[STEP] step={step} action={action_safe} reward={reward:.2f} '
        f'done={str(done).lower()} error={error_val}',
        flush=True,
    )

def log_end(success: bool, steps: int, score: float, rewards: List[float]) -> None:
    rewards_str = ','.join(f'{r:.2f}' for r in rewards)
    print(
        f'[END] success={str(success).lower()} steps={steps} '
        f'score={score:.2f} rewards={rewards_str}',
        flush=True,
    )


# ── Qwen2.5-VL generate — drop-in for OpenAI client.chat.completions.create ──
def _strip_think(text: str) -> str:
    """Strip <think>...</think> chain-of-thought blocks Qwen3 emits before JSON."""
    return re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()


def _call_model(messages: list) -> str:
    """One inference pass — equivalent to client.chat.completions.create() in gemini_inference.py.

    Uses tokenizer.apply_chat_template + tokenizer(text=...) instead of tokenizer,
    slices output by input_len to get only new tokens, strips think tags.
    torch.cuda.empty_cache() called here to prevent KV-cache fragmentation across steps.
    """
    torch.cuda.empty_cache()
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], padding=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=max(TEMPERATURE, 0.01),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs['input_ids'].shape[1]
    raw = tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True)
    return _strip_think(raw)


# ── Action parser — identical to inference_gemini_round2_schema_drift.py ──
def parse_llm_response(text: str) -> PipelineAction:
    payload = _extract_action_payload(text)
    if not payload:
        return FALLBACK_ACTION
    normalized = _normalize_action_payload(payload)
    try:
        return PipelineAction(**normalized)
    except Exception:
        return FALLBACK_ACTION


def _extract_action_payload(text: str) -> Optional[dict]:
    if not text:
        return None
    text = text.strip()
    if '```' in text:
        text = '\n'.join(l for l in text.split('\n') if not l.strip().startswith('```'))
    start = text.find('{')
    if start == -1:
        return None
    end = text.rfind('}') + 1
    if end > start:
        try:
            data = json.loads(text[start:end])
            if isinstance(data, dict) and 'action_type' in data:
                return data
        except Exception:
            pass
    repaired = _try_repair_json(text[start:])
    return repaired if isinstance(repaired, dict) else None


def _normalize_action_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    action_type = str(payload.get('action_type', '')).strip()
    params = payload.get('params') or {}
    if not isinstance(params, dict):
        params = {}
    return {'action_type': action_type, 'params': params}


def _try_repair_json(fragment: str) -> Optional[dict]:
    """Repair truncated JSON — handles the case where MAX_TOKENS cuts off mid-object."""
    for suffix in ['"}}', '}}', '}"}}', '}']:
        try:
            data = json.loads(fragment + suffix)
            if isinstance(data, dict) and 'action_type' in data:
                return data
        except Exception:
            continue
    at_match = re.search(r'"action_type"\s*:\s*"([^"]+)"', fragment)
    if not at_match:
        return None
    action_type = at_match.group(1)
    params: Dict[str, Any] = {}
    for key, pattern in [
        ('step_id',           r'"step_id"\s*:\s*"([^"]+)"'),
        ('patch_type',        r'"patch_type"\s*:\s*"([^"]+)"'),
        ('column',            r'"column"\s*:\s*"([^"]+)"'),
        ('table',             r'"table"\s*:\s*"([^"]+)"'),
        ('filter_condition',  r'"filter_condition"\s*:\s*"([^"]+)"'),
        ('team',              r'"team"\s*:\s*"([^"]+)"'),
        ('issue_description', r'"issue_description"\s*:\s*"([^"]+)"'),
        ('strategy',          r'"strategy"\s*:\s*"([^"]+)"'),
        ('old_column',        r'"old_column"\s*:\s*"([^"]+)"'),
        ('new_column',        r'"new_column"\s*:\s*"([^"]+)"'),
    ]:
        m = re.search(pattern, fragment)
        if m:
            params[key] = m.group(1)
    m = re.search(r'"n_rows"\s*:\s*(\d+)', fragment)
    if m:
        params['n_rows'] = int(m.group(1))
    return {'action_type': action_type, 'params': params}


print(f'Config ready. MAX_STEPS={MAX_STEPS}, MAX_TOKENS={MAX_TOKENS}, TEMPERATURE={TEMPERATURE}')


In [ ]:
# ── System prompt — identical to inference_gemini_round2_schema_drift.py ──────
SYSTEM_PROMPT = textwrap.dedent("""
You are an expert data engineer diagnosing and fixing broken data pipelines.

You will receive the current state of a pipeline (failing assertions, DAG structure,
historical run info) and must choose ONE action to take each turn.

WORKFLOW (follow this order strictly):
1. FIRST: read_data_sample on the raw input table(s) to see what the data looks like.
2. THEN: Use check_schema or compare_schema if a type or schema issue is suspected.
3. If you see any schema drift signal (renamed/missing columns, changed types, auth format drift,
   or stricter rate-limit behavior), use handle_drift.
4. THEN: Apply the RIGHT fix using add_data_filter or patch_transformation.
5. THEN: Call run_pipeline to verify the fix.
6. ONLY AFTER fixing what you can: If some data is genuinely corrupted (e.g. \"N/A\" values
   that cannot be parsed), call alert_upstream_team.
7. If assertions are still failing after run_pipeline, investigate more and apply
   additional fixes. Do NOT just call run_pipeline again without changing something.

AVAILABLE ACTIONS (respond with ONLY a JSON object, no markdown, no prose):

{\"action_type\": \"read_data_sample\", \"params\": {\"table\": \"<table_name>\", \"n_rows\": 20}}
{\"action_type\": \"check_schema\", \"params\": {\"table\": \"<table_name>\"}}
{\"action_type\": \"compare_schema\", \"params\": {\"table\": \"<table_name>\"}}
{\"action_type\": \"handle_drift\", \"params\": {\"strategy\": \"<detect|numeric_format|null_fill|type_cast|join_key_prefix|filter_invalid|resolve_column_rename|alert_upstream>\", \"table\": \"<table_name_optional>\", \"step_id\": \"<step_id_optional>\", \"column\": \"<column_optional>\", \"old_column\": \"<optional_old_name>\", \"new_column\": \"<optional_new_name>\", \"filter_condition\": \"<optional>\", \"team\": \"<optional>\", \"issue_description\": \"<optional>\"}}
{\"action_type\": \"run_quality_assertion\", \"params\": {\"assertion_id\": \"<e.g. A1>\"}}
{\"action_type\": \"add_data_filter\", \"params\": {\"step_id\": \"<step_id>\", \"filter_condition\": \"<e.g. user_id IS NOT NULL>\"}}
{\"action_type\": \"patch_transformation\", \"params\": {\"step_id\": \"<step_id>\", \"patch_type\": \"<cast_column|coalesce|dedup|parse_currency|strip_prefix>\", \"column\": \"<column_name>\"}}
{\"action_type\": \"backfill_partition\", \"params\": {\"date\": \"<YYYY-MM-DD>\"}}
{\"action_type\": \"alert_upstream_team\", \"params\": {\"team\": \"<team_name_snake_case>\", \"issue_description\": \"<short description>\"}}
{\"action_type\": \"mark_acceptable\", \"params\": {\"assertion_id\": \"<id>\", \"reason\": \"<reason>\"}}
{\"action_type\": \"run_pipeline\", \"params\": {}}

KEY PATCH TYPES:
- parse_currency: strips $, commas, casts to float, converts N/A to NaN.
  Apply to the RAW INPUT TABLE step (e.g. transform_insights), NOT downstream steps.
  After parse_currency, ALWAYS chain coalesce on the same column and step.
  Exception: if column is a denominator (e.g. impressions in CTR), use IS NOT NULL filter instead.
- coalesce: replace NaN with 0 AFTER parse_currency. Same step_id as parse_currency.
- cast_column: simple numeric cast. Only use if values are already clean numbers stored as strings.
- dedup: remove duplicate rows by key column. ONLY fix for uniqueness assertion failures.
- strip_prefix: strip spurious prefix (default CMP_) then chain cast_column if numeric.

STEP ID MAPPING (use these exact step_id values):
- raw_ads_insights -> transform_insights (fixes: parse_currency spend, parse_currency impressions)
- raw_conversions  -> transform_conversions (fixes: dedup purchase_id)
- campaign_id join key: strip_prefix on transform_insights or transform_conversions

DRIFT HANDLING:
- resolve_column_rename: use when compare_schema shows a column was renamed (e.g. spend->total_spend)
- join_key_prefix: use when campaign_id has CMP_ prefix mismatch between tables

UPSTREAM TEAM: always snake_case. Meta/Graph API issues -> meta_ads_api_team.

RULES:
- ONE JSON object only. No markdown, no prose.
- NEVER fix before reading data (-0.5 penalty).
- NEVER mark_acceptable (-1.0 penalty).
- NEVER repeat a failed action. Try something different.
- Do NOT call run_pipeline unless you applied a fix since the last run.
- After parse_currency, ALWAYS coalesce the same column before run_pipeline.
- For the hard/hard2 tasks: fix ALL tables before alerting. Fix order:
    1. parse_currency + coalesce on spend in transform_insights
    2. parse_currency on impressions in transform_insights, then IS NOT NULL filter
    3. dedup on purchase_id in transform_conversions
    4. strip_prefix on campaign_id in transform_insights, then cast_column
    5. run_pipeline
    6. alert_upstream_team meta_ads_api_team about N/A values in source data
""").strip()


# ── Drift signal collector — identical to Gemini script ─────────────
def _collect_schema_drift_signals(obs: PipelineObservation) -> List[str]:
    signals: List[str] = []
    desc = (obs.description or '').lower()
    if 'schema drift' in desc or 'contract' in desc:
        signals.append('Task description references schema/contract drift.')
    if obs.schema_diff:
        diff_json = json.dumps(obs.schema_diff).lower()
        if 'removed' in diff_json:
            signals.append('Historical columns appear removed in current schema.')
        if 'changed' in diff_json:
            signals.append('Column types differ from historical schema.')
        if 'new' in diff_json:
            signals.append('New columns detected relative to historical schema.')
    for r in obs.failed_assertions:
        actual = (r.actual or '').lower()
        if 'missing' in actual and 'column' in actual:
            signals.append(f'Assertion {r.assertion_id} reports a missing column.')
        if 'not found' in actual and 'column' in actual:
            signals.append(f'Assertion {r.assertion_id} reports a renamed or deleted column.')
        if 'type' in actual and ('object' in actual or 'string' in actual):
            signals.append(f'Assertion {r.assertion_id} indicates possible type drift.')
    deduped: List[str] = []
    for s in signals:
        if s not in deduped:
            deduped.append(s)
    return deduped[:6]


# ── Loop detector ────────────────────────────────────────────────────
def _detect_action_loop(actions_taken: List[str]) -> Optional[str]:
    if len(actions_taken) < 3:
        return None
    def _key(s: str) -> str:
        idx = s.find(']')
        return s[idx+1:].strip() if idx >= 0 else s.strip()
    last_3 = [_key(a) for a in actions_taken[-3:]]
    if last_3[0] == last_3[1] == last_3[2]:
        return last_3[0]
    if len(actions_taken) >= 4:
        last_4 = [_key(a) for a in actions_taken[-4:]]
        if last_4[0] == last_4[2] and last_4[1] == last_4[3]:
            return f'{last_4[0]} and {last_4[1]} (oscillating)'
    return None


def _build_loop_hint(obs: PipelineObservation, looped_action: str) -> str:
    hint = (
        f"\n[CRITICAL LOOP DETECTED]: You have been repeating '{looped_action}' "
        f"without progress. Try a COMPLETELY DIFFERENT action.\n"
    )
    for r in obs.failed_assertions:
        atype = (r.assertion_type or '').lower()
        col   = r.column or ''
        table = r.table or ''
        if atype == 'unique':
            hint += (
                f'\n  -> {r.assertion_id}: UNIQUENESS on "{col}" in "{table}". '
                '{"action_type": "patch_transformation", "params": {"step_id": "transform_' + table.replace('clean_','').replace('_clean','') + '", "patch_type": "dedup", "column": "' + col + '"}}'
            )
        elif atype == 'type_check':
            hint += (
                f'\n  -> {r.assertion_id}: TYPE CHECK on "{col}" in "{table}". '
                f'Use parse_currency on the RAW INPUT step for "{table}", then coalesce.'
            )
        elif atype == 'value_range':
            hint += (
                f'\n  -> {r.assertion_id}: VALUE RANGE on "{col}". '
                f'Apply coalesce after parse_currency on the same step.'
            )
        elif atype == 'row_count':
            hint += (
                f'\n  -> {r.assertion_id}: ROW COUNT on "{table}". '
                f'Likely join key mismatch or duplicates. Use compare_schema then strip_prefix.'
            )
    if 'run_pipeline' in looped_action:
        hint += '\n\n  Apply a fix BEFORE run_pipeline. Repeating run_pipeline does nothing.'
    # Hard-task specific: if stuck on insights/spend repeatedly
    if 'parse_currency' in looped_action and 'spend' in looped_action:
        hint += (
            '\n\n  [HARD TASK GUIDE]: parse_currency on spend is correct but is not enough alone.'
            '\n  You must ALSO fix ALL of these before run_pipeline will pass more assertions:'
            '\n  1. patch_transformation(transform_insights, parse_currency, impressions)'
            '\n  2. add_data_filter(transform_insights, impressions IS NOT NULL)'
            '\n  3. patch_transformation(transform_conversions, dedup, purchase_id)'
            '\n  4. patch_transformation(transform_insights, strip_prefix, campaign_id)'
            '\n  5. patch_transformation(transform_insights, cast_column, campaign_id)'
            '\n  6. run_pipeline'
            '\n  7. alert_upstream_team(meta_ads_api_team, ...)'
        )
    return hint


# ── Prompt builder ───────────────────────────────────────────────────
# KEY FIX: uses caller's max_steps not obs.max_steps
def build_user_prompt(obs: PipelineObservation, step: int, max_steps: int) -> str:
    failed_str = '\n'.join(
        f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}'
        for r in obs.failed_assertions
    ) or '  (none -- all passing!)'
    passed_str = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag_str = '\n'.join(
        f'  {n.step_id}: {n.input_table} -> {n.output_table}'
        + (f' | filters: {n.applied_filters}' if n.applied_filters else '')
        + (f' | patches: {n.applied_patches}' if n.applied_patches else '')
        for n in obs.dag_structure
    )
    hist_str = '\n'.join(
        f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs[-2:]
    )
    sample_str = ''
    if obs.data_sample:
        sample_rows = obs.data_sample[:5]
        null_rows   = [r for r in obs.data_sample if any(v is None for v in r.values())]
        if null_rows:
            sample_str = (
                '\nDATA SAMPLE (first 5 rows):\n'
                + json.dumps(sample_rows, indent=2, default=str)
                + f'\nROWS WITH NULL VALUES ({len(null_rows)} found):\n'
                + json.dumps(null_rows[:5], indent=2, default=str)
            )
        else:
            sample_str = '\nDATA SAMPLE (first 5 rows):\n' + json.dumps(sample_rows, indent=2, default=str)
    schema_str = ''
    if obs.current_schema:
        schema_str = '\nCURRENT SCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff:
        schema_str += '\nSCHEMA DIFF vs HISTORICAL: ' + json.dumps(obs.schema_diff)
    drift_signals = _collect_schema_drift_signals(obs)
    drift_str = ''
    if drift_signals:
        drift_str = '\nSCHEMA DRIFT SIGNALS:\n' + '\n'.join(f'  - {s}' for s in drift_signals)
    actions_str = '\n'.join(f'  {a}' for a in obs.actions_taken[-5:]) or '  (none yet)'

    # ── Hint engine ────────────────────────────────────────────────
    read_actions  = sum(1 for a in obs.actions_taken if 'read_data_sample' in a or 'check_schema' in a)
    fix_actions   = sum(1 for a in obs.actions_taken if 'add_data_filter' in a or 'patch_transformation' in a)
    mark_actions  = sum(1 for a in obs.actions_taken if 'mark_acceptable' in a)
    parse_done    = any('parse_currency' in a for a in obs.actions_taken)
    coalesce_done = any('coalesce' in a for a in obs.actions_taken)
    recent_runs   = sum(1 for a in obs.actions_taken[-3:] if 'run_pipeline' in a)

    hint_str = ''
    looped_action = _detect_action_loop(obs.actions_taken)
    if looped_action:
        hint_str += _build_loop_hint(obs, looped_action)
    elif read_actions >= 2 and fix_actions == 0:
        hint_str = (
            '\n[HINT]: You have already read the data. '
            'Apply a fix now using add_data_filter or patch_transformation, then run_pipeline.'
        )
    value_range_failing = any(
        r.assertion_type == 'value_range' and 'non-numeric' in r.actual
        for r in obs.failed_assertions
    )
    if parse_done and not coalesce_done and value_range_failing:
        hint_str += (
            "\n[CRITICAL]: parse_currency applied but value_range still failing. "
            'Chain coalesce on the same column and step_id now.'
        )
    if mark_actions >= 1:
        hint_str += '\n[WARNING]: NEVER use mark_acceptable. -1.0 penalty every time.'
    if recent_runs >= 2 and not obs.pipeline_passed:
        hint_str += (
            '\n[CRITICAL]: run_pipeline called multiple times with no progress. '
            'Apply a NEW fix before calling run_pipeline again.'
        )

    steps_remaining = max_steps - step

    return textwrap.dedent(f"""
    STEP {step}/{max_steps}  |  STEPS REMAINING: {steps_remaining}
    TASK: {obs.task_id} ({obs.difficulty})
    DESCRIPTION: {obs.description}
    PIPELINE PASSED: {obs.pipeline_passed}
    LAST ACTION RESULT: {obs.last_action_result}

    DAG STRUCTURE:
    {dag_str}

    FAILING ASSERTIONS:
    {failed_str}

    PASSING ASSERTIONS: {passed_str}

    HISTORICAL RUNS:
    {hist_str}

    RECENT ACTIONS TAKEN:
    {actions_str}
    {sample_str}{schema_str}{drift_str}{hint_str}

    Respond with exactly ONE action JSON object.
    """).strip()


print('System prompt, drift signals, loop detection, and prompt builder ready.')


In [ ]:
from src.environment import DataPipelineEnv


def run_episode(task_id: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> Dict[str, Any]:
    # Defensive creation: supports both old env (no max_steps) and new env
    try:
        env = DataPipelineEnv(task_id=task_id, max_steps=max_steps)
    except TypeError:
        env = DataPipelineEnv(task_id=task_id)
        env.MAX_STEPS = max_steps

    history:         List[Dict[str, str]] = []
    rewards:         List[float]          = []
    steps_taken:     int                  = 0
    score:           float                = 0.0
    success:         bool                 = False
    n_passed:        int                  = 0
    n_total:         int                  = 0
    pipeline_passed: bool                 = False

    log_start(task=task_id, env=BENCHMARK, model=MODEL_NAME)

    try:
        obs = env.reset()
        if verbose:
            print(f'\n{"="*60}', file=sys.stderr)
            print(f'TASK: {task_id.upper()}  |  {len(obs.failed_assertions)} assertions failing', file=sys.stderr)
            print(f'Running for max_steps={max_steps} (env internal={obs.max_steps})', file=sys.stderr)
            print(f'{"="*60}', file=sys.stderr)
            print(f'Description: {obs.description}', file=sys.stderr)

        for step in range(1, max_steps + 1):
            if obs.pipeline_passed:
                if verbose:
                    print(f'\n[PASSED] Pipeline passed at step {step - 1}!', file=sys.stderr)
                break

            # Loop detection: trim history to break repetitive context
            looped = _detect_action_loop(obs.actions_taken)
            if looped:
                if verbose:
                    print(f'  [Step {step}] Loop detected: {looped}. Trimming history.', file=sys.stderr)
                history = history[-2:]

            # NOTE: pass max_steps (our budget) to build_user_prompt, not obs.max_steps
            user_prompt = build_user_prompt(obs, step, max_steps)
            history.append({'role': 'user', 'content': user_prompt})
            messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history

            response_text = ''
            try:
                response_text = _call_model(messages)
            except torch.cuda.OutOfMemoryError:
                print(f'[OOM] Step {step}: trimming history to 4 turns and retrying.', file=sys.stderr)
                history = history[-4:]
                messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + history
                torch.cuda.empty_cache()
                try:
                    response_text = _call_model(messages)
                except Exception as e2:
                    print(f'[OOM-RETRY-FAIL] {e2}', file=sys.stderr)
            except Exception as exc:
                if verbose:
                    print(f'  [Step {step}] Generation error: {exc}. Using fallback.', file=sys.stderr, flush=True)

            action = parse_llm_response(response_text)

            # Smart fallback: empty response -> safe diagnostic instead of run_pipeline
            if action.action_type == 'run_pipeline' and not response_text.strip():
                if obs.failed_assertions:
                    action = PipelineAction(
                        action_type='compare_schema',
                        params={'table': obs.failed_assertions[0].table}
                    )

            history.append({'role': 'assistant', 'content': response_text or '{}'})
            if len(history) > 14:
                history = history[-14:]

            result = env.step(action)
            obs    = result.observation
            reward = result.reward or 0.0
            done   = result.done
            error: Optional[str] = getattr(obs, 'last_action_error', None) or None

            rewards.append(reward)
            steps_taken = step

            log_step(
                step=step,
                action=json.dumps(action.model_dump()).replace('\n', ' ')[:200],
                reward=reward,
                done=done,
                error=error,
            )

            if verbose:
                print(f'\n[Step {step}] Raw: {response_text[:120]}', file=sys.stderr)
                print(f'[Step {step}] Action: {action.action_type}({action.params})', file=sys.stderr)
                print(
                    f'  Reward: {reward:+.2f} | '
                    f'Passed: {len(obs.passed_assertions)}/{len(obs.failed_assertions)+len(obs.passed_assertions)} | '
                    f'Result: {obs.last_action_result[:80]}',
                    file=sys.stderr
                )

            if done:
                break

        n_total  = len(obs.failed_assertions) + len(obs.passed_assertions)
        n_passed = len(obs.passed_assertions)
        pipeline_passed = obs.pipeline_passed
        raw_score = n_passed / n_total if n_total > 0 else 0.0
        score   = min(max(raw_score, 0.01), 0.99)
        success = score >= SUCCESS_SCORE_THRESHOLD

        if verbose:
            print(f'\n--- Episode Summary ---', file=sys.stderr)
            print(f'  Score: {score:.2f}  Reward: {sum(rewards):.2f}  '
                  f'Steps: {steps_taken}/{max_steps}  Passed: {pipeline_passed}', file=sys.stderr)

    except Exception as exc:
        import traceback
        print(f'[ERROR] {task_id}: {exc}', file=sys.stderr)
        traceback.print_exc(file=sys.stderr)
    finally:
        try:
            env.close()
        except AttributeError:
            pass
        except Exception as e:
            print(f'[DEBUG] env.close() error: {e}', file=sys.stderr, flush=True)
        log_end(success=success, steps=steps_taken, score=score, rewards=rewards)

    return {
        'task_id':            task_id,
        'score':              round(score, 4),
        'success':            success,
        'pipeline_passed':    pipeline_passed,
        'total_reward':       round(sum(rewards), 4),
        'steps_taken':        steps_taken,
        'assertions_passed':  n_passed,
        'assertions_total':   n_total,
    }


print('Runner ready.')


In [ ]:
# Dynamically detect which tasks are available in the cloned branch
from src.tasks import TASKS as _AVAILABLE_TASKS

ALL_TASKS   = ['easy', 'medium', 'hard', 'hard2']
VALID_TASKS = [t for t in ALL_TASKS if t in _AVAILABLE_TASKS]
print(f'Available tasks in this branch: {VALID_TASKS}', file=sys.stderr)

# Change to run a single task: 'easy' | 'medium' | 'hard' | 'hard2' | 'all'
TASKS_TO_RUN = 'all'

task_ids = VALID_TASKS if TASKS_TO_RUN == 'all' else (
    [TASKS_TO_RUN] if TASKS_TO_RUN in _AVAILABLE_TASKS else []
)

all_results = []
for task_id in task_ids:
    result = run_episode(task_id=task_id, max_steps=MAX_STEPS, verbose=True)
    all_results.append(result)
    torch.cuda.empty_cache()

print('\n' + '='*60, file=sys.stderr)
print('FINAL SCORES', file=sys.stderr)
print('='*60, file=sys.stderr)
total_score = 0.0
for r in all_results:
    status = '[PASSED]' if r['pipeline_passed'] else '[FAILED]'
    print(
        f"  {r['task_id']:8s} | score={r['score']:.2f} | "
        f"reward={r['total_reward']:+.2f} | steps={r['steps_taken']:2d} | {status}",
        file=sys.stderr
    )
    total_score += r['score']

avg = total_score / len(all_results) if all_results else 0.0
print(f'\n  Avg score: {avg:.4f}', file=sys.stderr)

# JSON_RESULTS to stderr — keeps stdout clean for the OpenEnv spec parser
json_str = json.dumps(all_results, indent=2)
json_str = re.sub(r'"total_reward":\s*(-?\d+(?:\.\d+)?)',
                  lambda m: f'"total_reward": {float(m.group(1)):.2f}', json_str)
json_str = re.sub(r'"score":\s*(-?\d+(?:\.\d+)?)',
                  lambda m: f'"score": {float(m.group(1)):.2f}', json_str)
print('\nJSON_RESULTS:', json_str, file=sys.stderr)


In [16]:
# Run any time to check VRAM health
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total_vr  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM allocated : {allocated:.2f} GB')
print(f'VRAM reserved  : {reserved:.2f} GB')
print(f'VRAM free      : {total_vr - reserved:.2f} GB  (of {total_vr:.1f} GB)')


VRAM allocated : 3.53 GB
VRAM reserved  : 3.70 GB
VRAM free      : 11.94 GB  (of 15.6 GB)
